# Base model vs SFT — запуск обучения на Kaggle

**Перед запуском (Settings справа):**
- Accelerator → **GPU T4 x2** (используем одну)
- Internet → **On**
- (необязательно) Add-ons → Secrets → `HF_TOKEN`, `WANDB_API_KEY` — поставить галочки

**Как пользоваться:**
- Что запускать — меняется только в ячейке **5. Настройки запуска**.
- **Полное обучение (~50 мин):** `SMOKE = False` → *Save Version* → **Save & Run All (Commit)**. Идёт в фоне,
  вкладку можно закрыть. Всё из `/kaggle/working` (адаптер, `results/`, графики) сохранится в **Output** версии.
- **Отладка:** `SMOKE = True` → *Run All* в интерактивной сессии (пара минут).
- Сессия живая, код на GitHub обновился → ячейки 3, 6, 7, 8.

**Графики:** ячейка 7 обновляет их каждую минуту, пока идёт обучение (loss train/val, token accuracy,
learning rate, grad norm + хвост лога). В конце — итоговые графики в ячейке 8 и PNG в `results/`.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Секреты (необязательно)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
for name in ("HF_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[name] = secrets.get_secret(name)
        print(f"{name}: подключён")
    except Exception:
        print(f"{name}: не найден — продолжаем без него")

## 3. Код из GitHub
На новой сессии клонирует репозиторий, на живой — делает `git pull`.

In [ ]:
REPO_URL = "https://github.com/Pavel-V418/BaseModel-vs-SFT.git"
REPO_DIR = "/kaggle/working/BaseModel-vs-SFT"
BRANCH = "master"

import os
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git fetch -q && git checkout -q {BRANCH} && git pull -q
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
!git log --oneline -3

## 4. Библиотеки
`torch` уже стоит в Kaggle — не переустанавливаем. `torchao` удаляем: старая версия в образе Kaggle ломает свежий `peft`.

In [ ]:
!pip install -q -U trl peft bitsandbytes accelerate gdown
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao

In [ ]:
import torch, transformers, trl, peft, datasets
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("transformers", transformers.__version__, "| trl", trl.__version__,
      "| peft", peft.__version__, "| datasets", datasets.__version__)

## 4.1. Данные Spider
Скачивание + подготовка `data/{train,val,test,fewshot}.jsonl` (несколько минут). Если данные уже есть в этой сессии — пропускается.
Последний шаг `download_spider.sh` (проверка через CLI `sqlite3`) на Kaggle может упасть — это не критично.

In [ ]:
import json
import os

# Удаляем старые сгенерированные данные, чтобы принудительно пересобрать их с новыми примерами
if os.path.exists("data/train.jsonl"):
    !rm data/*.jsonl data/stats.json

!bash scripts/download_spider.sh data/spider
!python -m src.data --spider_dir data/spider --out_dir data --val_size 400 --seed 42 --include_others

st = json.load(open("data/stats.json"))
for name, s in st["splits"].items():
    tok = s.get("tokens_prompt_plus_completion", {})
    over = {k: v for k, v in tok.items() if k.startswith("n_over_")}
    print(f"{name:8s} примеров: {s['n_examples']:5d} | токенов: mean {tok.get('mean')}, "
          f"p95 {tok.get('p95')}, max {tok.get('max')} | длиннее max_length: {over}")

## 5. Настройки запуска
- `SMOKE = True` — проверка на 12 игрушечных примерах, 10 шагов, без W&B (~пара минут).
- `SMOKE = False` — настоящее обучение на `data/train.jsonl` и `data/val.jsonl`.
- `OVERRIDES` — поменять параметры конфига без правки файла, например:
  `["train_size=2000", "lora.r=32", "run_name=sft_0.5b_n2000_r32_lr1e-4_s42"]`
- `REFRESH_SEC` — как часто перерисовывать графики во время обучения.

In [ ]:
CONFIG = "configs/sft_0.5b.yaml"
SMOKE = False
OVERRIDES = [
    "run_name=sft_0.5b_nfull_others_r16_lr1e-4_s42",
    "training.num_train_epochs=1"
]
REFRESH_SEC = 60

import os
# без W&B-ключа отключаем логирование, чтобы запуск не завис на логине
if not SMOKE and "WANDB_API_KEY" not in os.environ:
    OVERRIDES = OVERRIDES + ["report_to=none"]
    print("WANDB_API_KEY нет — report_to=none (графики всё равно будут в ноутбуке)")

from src.utils import load_config
RUN_NAME = ("smoke_" if SMOKE else "") + load_config(CONFIG, OVERRIDES)["run_name"]
LIVE_PATH = f"results/{RUN_NAME}_live.json"     # пишет train.py на каждом шаге логирования
LOG_PATH = f"results/{RUN_NAME}_stdout.log"     # весь вывод обучения
print("Запуск:", RUN_NAME)

## 6. Запуск обучения (в фоне)
Обучение идёт отдельным процессом, весь вывод — в `results/<run>_stdout.log`.
Остановить обучение: `proc.terminate()` в новой ячейке.

In [ ]:
import subprocess, sys, time

cmd = [sys.executable, "-m", "src.train", "--config", CONFIG]
if SMOKE:
    cmd.append("--smoke")
for o in OVERRIDES:
    cmd += ["--set", o]
print(" ".join(cmd))

os.makedirs("results", exist_ok=True)
if os.path.exists(LIVE_PATH):
    os.remove(LIVE_PATH)  # чтобы не рисовать старый запуск с тем же именем

# одна GPU: на T4 x2 без CUDA_VISIBLE_DEVICES модель распределится по двум картам
env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "PYTHONUNBUFFERED": "1"}
log_file = open(LOG_PATH, "w")
proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, env=env)
T_START = time.time()
print("PID:", proc.pid)

## 7. Мониторинг: графики во время обучения
Перерисовывается каждые `REFRESH_SEC` секунд, пока процесс жив.
Прервать ячейку (■) можно в любой момент — **обучение продолжится**, графики вернёт повторный запуск этой ячейки.

In [ ]:
import json, re
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from src.plots import plot_history

def tail(path, n=12):
    try:
        text = open(path, encoding="utf-8", errors="replace").read()
    except FileNotFoundError:
        return ""
    lines = [l for l in re.split(r"[\r\n]+", text) if l.strip()]  # \r — от tqdm
    return "\n".join(lines[-n:])

def fmt(sec):
    sec = int(sec)
    return f"{sec // 3600}:{sec % 3600 // 60:02d}:{sec % 60:02d}"

_seen = [(0, time.time())]   # (шаг, время, когда мы его увидели) — для оценки скорости

def show():
    clear_output(wait=True)
    elapsed = time.time() - T_START
    status = "идёт" if proc.poll() is None else f"завершён (код {proc.returncode})"
    head = f"{RUN_NAME} | процесс: {status} | прошло {fmt(elapsed)}"
    live = None
    try:
        live = json.load(open(LIVE_PATH))
        steps = [r["step"] for r in live["history"] if r.get("phase") == "train"]
        step, total = (max(steps) if steps else 0), live.get("max_steps") or 0
        if step and total and proc.poll() is None:
            # скорость считаем по шагам обучения, без загрузки модели и оценки base
            now = time.time()
            if step != _seen[-1][0]:
                _seen.append((step, now))
            (s0, t0), (s1, t1) = _seen[1], _seen[-1]
            if s1 > s0:
                sec_per_step = (t1 - t0) / (s1 - s0)
                head += (f" | шаг {step}/{total}, {sec_per_step:.1f} с/шаг, "
                         f"осталось ~{fmt(sec_per_step * (total - step))}")
            else:
                head += f" | шаг {step}/{total}, скорость считается..."
    except (FileNotFoundError, json.JSONDecodeError):
        pass
    print(head)
    if live is None:
        print("(ждём первых логов: загрузка модели и оценка base)")
    else:
        try:
            fig = plot_history(live)
            display(fig)
            plt.close(fig)
        except Exception as e:  # не роняем мониторинг из-за графика
            print(f"(графики пока недоступны: {e})")
    print("\n--- последние строки лога ---")
    print(tail(LOG_PATH))

try:
    while proc.poll() is None:
        show()
        time.sleep(REFRESH_SEC)
except KeyboardInterrupt:
    print("Мониторинг остановлен. Обучение продолжается в фоне — запусти ячейку ещё раз, чтобы вернуть графики.")
else:
    show()
    log_file.close()
    if proc.returncode != 0:
        raise RuntimeError(f"Обучение упало (код {proc.returncode}). Полный лог: {LOG_PATH}")

## 8. Результат и итоговые графики

In [ ]:
import json
from IPython.display import Image

r = json.load(open(f"results/{RUN_NAME}_train.json"))
print("Запуск:          ", r["run_name"])
print("train / val:     ", r["train_size"], "/", r["val_size"])
print("eval_loss до:    ", round(r["eval_before"]["eval_loss"], 4))
print("eval_loss после: ", round(r["eval_after"]["eval_loss"], 4))
p = r["params"]
print(f"обучаемых параметров: {p['trainable']:,} из {p['total']:,} ({100*p['trainable']/p['total']:.2f}%)")
print("время обучения:   ", round(r["train_time_sec"] / 60, 1), "мин")
print("лучший чекпоинт: ", r["best_checkpoint"])
print("адаптер:         ", r["adapter_dir"])
!ls -la {r["adapter_dir"]}

# проверка EOS: в примере после токенизации в конце должен быть <|endoftext|>, без «ВНИМАНИЕ»
log = open(LOG_PATH, encoding="utf-8", errors="replace").read()
print("\nEOS в конце примера:", "<|endoftext|>" in log.split("=== Пример после токенизации ===")[-1][:5000],
      "| предупреждение про EOS:", "последний токен не EOS" in log)

png = f"results/{RUN_NAME}_curves.png"
if not os.path.exists(png):  # на случай, если train.py не смог построить
    plot_history(LIVE_PATH, out=png)
display(Image(png))

## 9. (Необязательно) Скачать результаты
В режиме **Save & Run All (Commit)** всё из `/kaggle/working` и так попадёт в **Output** версии.
В интерактивной сессии файлы пропадут после её окончания — тогда скачай архив (появится справа в **Output**).

In [ ]:
from IPython.display import FileLink
%cd /kaggle/working
!zip -q -r results_{RUN_NAME}.zip BaseModel-vs-SFT/results BaseModel-vs-SFT/outputs/{RUN_NAME} -x "*/checkpoint-*"
!ls -la /kaggle/working/*.zip
%cd {REPO_DIR}
FileLink(f"results_{RUN_NAME}.zip")